In [23]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import pathlib
from os import path
from pathlib import Path
from datetime import datetime

In [28]:
def load_all_pdfs(pdf_directory):
    all_documents=[]
    pdf_dir=Path(pdf_directory)
    pdf_file=list(pdf_dir.glob("**/*.pdf"))
    for file in pdf_file:
        print(f"processing {file.name}")
        try:
            loader=PyMuPDFLoader(file)
            documents=loader.load()
            
            for doc in documents:
                doc.metadata['source_file']=file.name
                doc.metadata['type']='pdf'
                doc.metadata['creationdate']=datetime.now().isoformat()
                doc.metadata['creator']="One and only Manish"
                doc.metadata['producer']="LPU"
            all_documents.extend(documents)
            print(f" loaded:{len(documents)} pages")
        except FileNotFoundError as e:
            print("file not found ")
            return None
        print(f"Numbers of files {len(pdf_file)} loaded")
    return all_documents
    
all_document=load_all_pdfs("../data")
all_document


processing DECAP737_MACHINE LEARNING.pdf
 loaded:187 pages
Numbers of files 2 loaded
processing Rag.pdf
 loaded:21 pages
Numbers of files 2 loaded


[Document(metadata={'producer': 'LPU', 'creator': 'One and only Manish', 'creationdate': '2026-09-23T12:05:04.084222', 'source': '..\\data\\DECAP737_MACHINE LEARNING.pdf', 'file_path': '..\\data\\DECAP737_MACHINE LEARNING.pdf', 'total_pages': 187, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-07-25T09:47:43+05:30', 'trapped': '', 'modDate': "D:20230725094743+05'30'", 'creationDate': '', 'page': 0, 'source_file': 'DECAP737_MACHINE LEARNING.pdf', 'type': 'pdf'}, page_content='Machine Learning\nDECAP737\nEdited by\nDr. V Devenderan'),
 Document(metadata={'producer': 'LPU', 'creator': 'One and only Manish', 'creationdate': '2026-09-23T12:05:04.084233', 'source': '..\\data\\DECAP737_MACHINE LEARNING.pdf', 'file_path': '..\\data\\DECAP737_MACHINE LEARNING.pdf', 'total_pages': 187, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-07-25T09:47:43+05:30', 'trapped': '', 'modDate': "D:20230725094743+0

In [29]:
def spilt_document(documents,chunk_size=1000,chunk_overlap=100):
    splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
chunks=spilt_document(all_document)

Split 208 documents into 540 chunks

Example chunk:
Content: Machine Learning
DECAP737
Edited by
Dr. V Devenderan...
Metadata: {'producer': 'LPU', 'creator': 'One and only Manish', 'creationdate': '2026-09-23T12:05:04.084222', 'source': '..\\data\\DECAP737_MACHINE LEARNING.pdf', 'file_path': '..\\data\\DECAP737_MACHINE LEARNING.pdf', 'total_pages': 187, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-07-25T09:47:43+05:30', 'trapped': '', 'modDate': "D:20230725094743+05'30'", 'creationDate': '', 'page': 0, 'source_file': 'DECAP737_MACHINE LEARNING.pdf', 'type': 'pdf'}


In [30]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [45]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLm-l6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        
        try:
            print(f"loading model {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"{self.model_name} loaded succesfully")
            print(f"Dimension of model{self.model.get_embedding_dimension} ")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self,texts:list[any]):
        
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embeddings=EmbeddingManager()
embeddings
            
            
        

loading model all-MiniLm-l6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2939.85it/s]


all-MiniLm-l6-v2 loaded succesfully
Dimension of model<bound method SentenceTransformer.get_embedding_dimension of SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)> 


In [46]:
import os
import uuid
import numpy as np
import chromadb
from pathlib import Path
from typing import List, Any


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                embeddings=embeddings_list,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [51]:
chunks=spilt_document(all_document)
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

document_embeddings=embeddings.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,document_embeddings)

Split 208 documents into 540 chunks

Example chunk:
Content: Machine Learning
DECAP737
Edited by
Dr. V Devenderan...
Metadata: {'producer': 'LPU', 'creator': 'One and only Manish', 'creationdate': '2026-09-23T12:05:04.084222', 'source': '..\\data\\DECAP737_MACHINE LEARNING.pdf', 'file_path': '..\\data\\DECAP737_MACHINE LEARNING.pdf', 'total_pages': 187, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-07-25T09:47:43+05:30', 'trapped': '', 'modDate': "D:20230725094743+05'30'", 'creationDate': '', 'page': 0, 'source_file': 'DECAP737_MACHINE LEARNING.pdf', 'type': 'pdf'}
Generating embeddings for 540 texts...


Batches: 100%|██████████| 17/17 [00:11<00:00,  1.51it/s]


Generated embeddings with shape: (540, 384)
Adding 540 documents to vector store...
Successfully added 540 documents
Total documents in collection: 540


In [52]:
len(all_document)

208